In [0]:
%sql
drop table if exists real_estate.silver.propiedades;

create table real_estate.silver.propiedades (
    propiedad_id bigint generated always as identity (start with 1 increment by 1) comment 'ID único de la propiedd (auto-generado)',

    partido string comment 'Partido/municipio estandarizado',
    region string comment 'Región geográfica: capital federal, gba zona norte/oeste/sur',

    tipo_de_operacion string comment 'alquiler, venta, alquiler_temporario',
    precio decimal(15,2) comment 'Precio de la propiedad',
    moneda string comment 'USD O ARS',
    expensas decimal(15,2) comment 'Expensas mensuales',

    ambientes int comment 'Cantidad de ambientes',
    metros_cuadrados_totales decimal(15,2) comment 'Superficie total',
    metros_cuadrados_cubiertos decimal(15,2) comment 'Superficie cubierta',

    antiguedad int comment 'Años de antiguedad (999 si no disponible)',
    cochera boolean comment 'Tiene cochera',
    orientacion string comment 'Orientación del inmueble',
    estado string comment 'Estado de la propiedad',
    url string comment 'URL de la propiedad',

    fecha_publicacion date comment 'Fecha de publicación — desde fecha (STRING) en Bronze',

    precio_por_m2 decimal(15,2) comment 'precio por metro cuadrado',

    _source_table string default 'real_estate.bronze.properties_bronze' comment 'Tabla origen',
    _processing_timestamp timestamp default current_timestamp comment 'Timestamp de procesamiento'
 )
 using delta 
 TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
 COMMENT 'Propiedades inmobiliarias - Capa Silver (datos limpios y validados)';

In [0]:
%sql
describe table extended real_estate.silver.propiedades;

In [0]:
%sql
-- propiedades_clean: casting básico de tipos 
create or replace temporary view propiedades_clean as 
select 
    case when precio rlike '^[^a-zA-Z]+$' then precio::double else null end as precio,
    moneda,
    case when ambientes rlike '^[^a-zA-Z]+$' then ambientes::double else null end as ambientes,
    case when metros_cuadrados_totales rlike '^[^a-zA-Z]+$' then metros_cuadrados_totales::double else null end as metros_cuadrados_totales,
    case when metros_cuadrados_cubiertos rlike '^[^a-zA-Z]+$' then metros_cuadrados_cubiertos::double else null end as metros_cuadrados_cubiertos,
    case when antiguedad rlike '^[^a-zA-Z]+$' then antiguedad::double else null end as antiguedad,
    tipo_de_operacion, id, ubicacion, numero, calle, expensas,
    orientacion_cardinal, orientacion_inmueble, piso, cochera, estado,
    tipo_vendedor, url, zona, fecha, hora
from real_estate.bronze.properties_bronze;

-- propiedades_clean_2: filtro de outliers por percentiles (Semana 2)
create or replace temporary view propiedades_clean_2 as 
(
  with limites as (
    select moneda, tipo_de_operacion,
      percentile(precio, 0.01) as p01,
      percentile(precio, 0.99) as p99
    from propiedades_clean
    where precio > 0 
      and moneda in ('ARS', 'USD')
      and tipo_de_operacion in ('venta', 'alquiler')
    group by moneda, tipo_de_operacion
  )
  select p.*
  from propiedades_clean p 
  join limites l
  on p.moneda = l.moneda
  and p.tipo_de_operacion = l.tipo_de_operacion
  where p.precio between l.p01 and l.p99
);

-- bronze_EDA: limpieza detallada 
create or replace temporary view bronze_EDA as (
    select 
        edl.id,
        edl.ubicacion,
        case 
          when edl.precio = 'NaN' or edl.precio not rlike '^-?[0-9]+\\.?[0-9]*$' then null
          when edl.precio::float between 0 and 2147483648 then edl.precio::float 
          else null
        end as precio,
        case 
          when edl.numero = 'NaN' or edl.numero not rlike '^-?[0-9]+\\.?[0-9]*$' then null
          when edl.numero::float between -10000 and 50000 then edl.numero::float
          else null
        end as numero,
        case 
          when edl.expensas = 'NaN' or edl.expensas not rlike '^-?[0-9]+\\.?[0-9]*$' then null 
          when edl.expensas::float  between 0 and 2000000 then edl.expensas::float 
          else null 
        end as expensas,
        edl.tipo_de_operacion,
        case 
          when lower(edl.moneda) like '%dolares%' then 'USD'
          when lower(edl.moneda) like '%us%' then 'USD'
          when lower(edl.moneda) like '%pesos%' then 'ARS'
          when lower(edl.moneda) like '%ars' then 'ARS'
          else edl.moneda
        end as moneda,
        case 
          when edl.ambientes = 'NaN' or edl.ambientes not rlike '^-?[0-9]+\\.?[0-9]*$' then null 
          else edl.ambientes::float
        end as ambientes,
        case 
          when edl.metros_cuadrados_totales = 'NaN' or edl.metros_cuadrados_totales not rlike '^-?[0-9]+\\.?[0-9]*$' then null
          else edl.metros_cuadrados_totales::float 
        end as metros_cuadrados_totales,
        case
          when edl.metros_cuadrados_cubiertos = 'NaN' or edl.metros_cuadrados_cubiertos not rlike '^-?[0-9]+\\.?[0-9]*$' then null
          else edl.metros_cuadrados_cubiertos::float
        end as metros_cuadrados_cubiertos,
        edl.orientacion_cardinal,
        edl.orientacion_inmueble,
        case 
          when edl.piso is null then null
          when edl.piso = 'NaN' then null
          else edl.piso::float
        end as piso,
        case 
          when edl.cochera = 'tiene' then 1 
          else null 
        end as cochera,
        edl.antiguedad,
        edl.estado,
        edl.tipo_vendedor,
        edl.url,
        edl.zona,
        coalesce(cast(edl.fecha as date), current_date()) as fecha
    from propiedades_clean_2 edl
    where edl.url rlike 'https'
      and (edl.zona like '%gba%' or edl.zona like '%caba%' or edl.zona like '%capital%')
      and edl.zona like '%-%'
      and length(edl.zona) < 50
      
);

select count(*) as registros_bronze_eda from bronze_EDA


### CTE datos_tipados (verificar tipos)

In [0]:
%sql

with datos_tipados as (
  select 
    lower(trim(zona)) as zona,
    lower(trim(tipo_de_operacion)) as tipo_de_operacion,
    cast(precio as decimal(15,2)) as precio,
    moneda,
    coalesce(cast(expensas as decimal(15,2)), 0) as expensas,
    cast(ambientes as int) as ambientes,
    cast(metros_cuadrados_totales as decimal(15,2)) as metros_cuadrados_totales,
    cast(metros_cuadrados_cubiertos as decimal(15,2)) as metros_cuadrados_cubiertos,
    case
      when antiguedad rlike '^[0-9]' and cast(antiguedad as float) between 0 and 200
           then cast(antiguedad as int)
           else 999
    end as antiguedad, 
    case when cochera = 1 then true else false end as cochera,
    coalesce(trim(orientacion_inmueble), '-1') as orientacion,
    coalesce(trim(estado), '-1') as estado,
    url,
    fecha as fecha_publicacion
from bronze_EDA
where precio > 0
  and metros_cuadrados_totales > 0
  and tipo_de_operacion is not null
)
select *, typeof(precio) as tipo_precio, typeof(ambientes) as tipo_ambientes
from datos_tipados
limit 10;

### CTE datos_deduplicados (contar duplicados)

In [0]:
%sql
with datos_tipados as (
    select 
        lower(trim(zona)) as zona,
        lower(trim(tipo_de_operacion)) as tipo_operacion,
        cast(precio as decimal(15,2)) as precio,
        moneda,
        coalesce(cast(expensas as decimal(15,2)), 0) as expensas,
        cast(ambientes as int) as ambientes,
        cast(metros_cuadrados_totales as decimal(15,2)) as metros_cuadrados_totales,
        cast(metros_cuadrados_cubiertos as decimal(15,2)) as metros_cuadrados_cubiertos,
        case 
          when antiguedad rlike '^[0-9]' and cast(antiguedad as float) between 0 and 200 
               then cast(antiguedad as int)
          else 999
        end as antiguedad,
        case when cochera = 1 then true else false end as cochera,
        coalesce(trim(orientacion_inmueble), '-1') as orientacion,
        coalesce(trim(estado), '-1') as estado,
        url,
        fecha as fecha_publicacion
    from bronze_EDA
    where precio > 0
      and metros_cuadrados_totales > 0
      and tipo_de_operacion is not null
),
datos_deduplicados as (
    select *,
           row_number() over (partition by precio, url order by precio asc) as rn
    from datos_tipados
)
select
    count(*) as total,
    count(case when rn = 1 then 1 end) as unicos,
    count(case when rn > 1 then 1 end) as duplicados
from datos_deduplicados;

In [0]:
%sql
insert into real_estate.silver.propiedades (
    partido,
    region,
    tipo_de_operacion,
    precio,
    moneda,
    expensas,
    ambientes,
    metros_cuadrados_totales,
    metros_cuadrados_cubiertos,
    antiguedad,
    cochera,
    orientacion,
    estado,
    url,
    fecha_publicacion,
    precio_por_m2,
    _source_table,
    _processing_timestamp
)

-- CTE 1: Tipos finales desde bronze_EDA
with datos_tipados as (
    select 
        lower(trim(zona)) as zona,
        lower(trim(tipo_de_operacion)) as tipo_de_operacion,
        cast(precio as decimal(15,2)) as precio,
        moneda,
        coalesce(cast(expensas as decimal(15,2)), 0) as expensas,
        cast(ambientes as int) as ambientes,
        cast(metros_cuadrados_totales as decimal(15,2)) as metros_cuadrados_totales,
        cast(metros_cuadrados_cubiertos as decimal(15,2)) as metros_cuadrados_cubiertos,
        case 
          when antiguedad rlike '^[0-9]' and cast(antiguedad as float) between 0 and 200 
               then cast(antiguedad as int) 
           else 999
        end as antiguedad,
        case when cochera = 1 then true else false end as cochera,
        coalesce(trim(orientacion_inmueble), '-1') as orientacion,
        coalesce(trim(estado), '-1') as estado,
        url,
        fecha as fecha_publicacion
        from bronze_EDA
        where precio > 0 
          and metros_cuadrados_totales > 0
          and tipo_de_operacion is not null
),
-- CTE 2: Deduplicación
datos_deduplicados as (
    select *,
        row_number() over (partition by url, precio order by precio) as rn
    from datos_tipados
),
datos_finales AS (
    SELECT
        CASE
            WHEN zona = 'capital-federal' THEN 'capital federal'
            WHEN zona = 'bsas-gba-norte/vicente-lopez' THEN 'vicente lopez'
            WHEN zona = 'bsas-gba-norte/pilar' THEN 'pilar'
            WHEN zona = 'bsas-gba-norte/tigre' THEN 'tigre'
            WHEN zona = 'bsas-gba-norte/san-isidro' THEN 'san isidro'
            WHEN zona = 'bsas-gba-norte/general-san-martin' THEN 'general san martin'
            WHEN zona = 'bsas-gba-norte/san-fernando' THEN 'san fernando'
            WHEN zona = 'bsas-gba-norte/san-miguel' THEN 'san miguel'
            WHEN zona = 'bsas-gba-norte/malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona = 'bsas-gba-norte/escobar' THEN 'escobar'
            WHEN zona = 'bsas-gba-oeste/caseros' THEN 'tres de febrero'
            WHEN zona = 'bsas-gba-oeste/castelar' THEN 'moron'
            WHEN zona = 'bsas-gba-oeste/general-rodriguez' THEN 'general rodriguez'
            WHEN zona = 'bsas-gba-oeste/hurlingham' THEN 'hurlingham'
            WHEN zona = 'bsas-gba-oeste/ituzaingo' THEN 'ituzaingo'
            WHEN zona = 'bsas-gba-oeste/la-matanza' THEN 'la matanza'
            WHEN zona = 'bsas-gba-oeste/merlo' THEN 'merlo'
            WHEN zona = 'bsas-gba-oeste/moreno' THEN 'moreno'
            WHEN zona = 'bsas-gba-oeste/moron' THEN 'moron'
            WHEN zona = 'bsas-gba-oeste/tres-de-febrero' THEN 'tres de febrero'
            WHEN zona = 'bsas-gba-sur/almirante-brown' THEN 'almirante brown'
            WHEN zona = 'bsas-gba-sur/avellaneda' THEN 'avellaneda'
            WHEN zona = 'bsas-gba-sur/berazategui' THEN 'berazategui'
            WHEN zona = 'bsas-gba-sur/esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona = 'bsas-gba-sur/ezeiza' THEN 'ezeiza'
            WHEN zona = 'bsas-gba-sur/florencio-varela' THEN 'florencio varela'
            WHEN zona = 'bsas-gba-sur/lanus' THEN 'lanus'
            WHEN zona = 'bsas-gba-sur/la-plata' THEN 'la plata'
            WHEN zona = 'bsas-gba-sur/lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona = 'bsas-gba-sur/quilmes' THEN 'quilmes'
            WHEN zona RLIKE 'vicente-lopez' THEN 'vicente lopez'
            WHEN zona RLIKE 'pilar' THEN 'pilar'
            WHEN zona RLIKE 'general-san-martin' THEN 'general san martin'
            WHEN zona RLIKE 'tigre' THEN 'tigre'
            WHEN zona RLIKE 'san-fernando' THEN 'san fernando'
            WHEN zona RLIKE 'san-miguel' THEN 'san miguel'
            WHEN zona RLIKE 'malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona RLIKE 'escobar' THEN 'escobar'
            WHEN zona RLIKE 'caseros' THEN 'tres de febrero'
            WHEN zona RLIKE 'castelar' THEN 'moron'
            WHEN zona RLIKE 'general-rodriguez' THEN 'general rodriguez'
            WHEN zona RLIKE 'hurlingham' THEN 'hurlingham'
            WHEN zona RLIKE 'ituzaingo' THEN 'ituzaingo'
            WHEN zona RLIKE 'la-matanza' THEN 'la matanza'
            WHEN zona RLIKE 'merlo' THEN 'merlo'
            WHEN zona RLIKE 'moreno' THEN 'moreno'
            WHEN zona RLIKE 'moron' THEN 'moron'
            WHEN zona RLIKE 'tres-de-febrero' THEN 'tres de febrero'
            WHEN zona RLIKE 'almirante-brown' THEN 'almirante brown'
            WHEN zona RLIKE 'avellaneda' THEN 'avellaneda'
            WHEN zona RLIKE 'berazategui' THEN 'berazategui'
            WHEN zona RLIKE 'berisso' THEN 'berisso'
            WHEN zona RLIKE 'ensenada' THEN 'ensenada'
            WHEN zona RLIKE 'esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona RLIKE 'ezeiza' THEN 'ezeiza'
            WHEN zona RLIKE 'florencio-varela' THEN 'florencio varela'
            WHEN zona RLIKE 'lanus' THEN 'lanus'
            WHEN zona RLIKE 'la-plata' THEN 'la plata'
            WHEN zona RLIKE 'lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona RLIKE 'presidente-peron' THEN 'presidente peron'
            WHEN zona RLIKE 'quilmes' THEN 'quilmes'
            WHEN zona RLIKE 'san-vicente' THEN 'san vicente'
            WHEN zona = 'san-isidro' THEN 'san isidro'
            WHEN zona = 'gba-zona-norte--vicente-lopez' THEN 'vicente lopez'
            WHEN zona = 'gba-zona-norte--pilar' THEN 'pilar'
            WHEN zona = 'gba-zona-norte--general-san-martin' THEN 'general san martin'
            WHEN zona = 'gba-zona-norte--san-isidro' THEN 'san isidro'
            WHEN zona = 'gba-zona-norte--tigre' THEN 'tigre'
            WHEN zona = 'gba-zona-norte--san-fernando' THEN 'san fernando'
            WHEN zona = 'gba-zona-norte--san-miguel' THEN 'san miguel'
            WHEN zona = 'gba-zona-norte--malvinas-argentinas' THEN 'malvinas argentinas'
            WHEN zona = 'gba-zona-norte--escobar' THEN 'escobar'
            WHEN zona = 'gba-zona-oeste--canuelas' THEN 'canuelas'
            WHEN zona = 'gba-zona-oeste--general-rodriguez' THEN 'general rodriguez'
            WHEN zona = 'gba-zona-oeste--hurlingham' THEN 'hurlingham'
            WHEN zona = 'gba-zona-oeste--ituzaingo' THEN 'ituzaingo'
            WHEN zona = 'gba-zona-oeste--la-matanza' THEN 'la matanza'
            WHEN zona = 'gba-zona-oeste--lujan' THEN 'lujan'
            WHEN zona = 'gba-zona-oeste--marcos-paz' THEN 'marcos paz'
            WHEN zona = 'gba-zona-oeste--merlo' THEN 'merlo'
            WHEN zona = 'gba-zona-oeste--moreno' THEN 'moreno'
            WHEN zona = 'gba-zona-oeste--moron' THEN 'moron'
            WHEN zona = 'gba-zona-oeste--tres-de-febrero' THEN 'tres de febrero'
            WHEN zona = 'gba-zona-sur--almirante-brown' THEN 'almirante brown'
            WHEN zona = 'gba-zona-sur--avellaneda' THEN 'avellaneda'
            WHEN zona = 'gba-zona-sur--berazategui' THEN 'berazategui'
            WHEN zona = 'gba-zona-sur--ensenada' THEN 'ensenada'
            WHEN zona = 'gba-zona-sur--san-vicente' THEN 'san vicente'
            WHEN zona = 'gba-zona-sur--esteban-echeverria' THEN 'esteban echeverria'
            WHEN zona = 'gba-zona-sur--ezeiza' THEN 'ezeiza'
            WHEN zona = 'gba-zona-sur--florencio-varela' THEN 'florencio varela'
            WHEN zona = 'gba-zona-sur--lanus' THEN 'lanus'
            WHEN zona = 'gba-zona-sur--la-plata' THEN 'la plata'
            WHEN zona = 'gba-zona-sur--lomas-de-zamora' THEN 'lomas de zamora'
            WHEN zona = 'gba-zona-sur--presidente-peron' THEN 'presidente peron'
            WHEN zona = 'gba-zona-sur--quilmes' THEN 'quilmes'
            ELSE 'no especifica'
        END as partido,

        CASE
            WHEN zona IS NULL THEN 'no especifica'
            WHEN zona = 'capital-federal' THEN 'capital federal'
            WHEN zona LIKE 'bsas-gba-norte/%' THEN 'gba zona norte'
            WHEN zona IN ('vicente-lopez','pilar','tigre','san-isidro','general-san-martin',
                         'san-fernando','san-miguel','jose-c-paz','malvinas-argentinas','escobar') THEN 'gba zona norte'
            WHEN zona LIKE 'bsas-gba-oeste/%' THEN 'gba zona oeste'
            WHEN zona IN ('caseros','castelar','general-rodriguez','hurlingham','ituzaingo',
                         'la-matanza','merlo','moreno','moron','tres-de-febrero') THEN 'gba zona oeste'
            WHEN zona LIKE 'bsas-gba-sur/%' THEN 'gba zona sur'
            WHEN zona IN ('almirante-brown','avellaneda','berazategui','berisso','ensenada',
                         'esteban-echeverria','ezeiza','florencio-varela','lanus','la-plata',
                         'lomas-de-zamora','presidente-peron','quilmes','san-vicente') THEN 'gba zona sur'
            WHEN zona LIKE 'gba-zona-norte--%' THEN 'gba zona norte'
            WHEN zona LIKE 'gba-zona-oeste--%' THEN 'gba zona oeste'
            WHEN zona LIKE 'gba-zona-sur--%' THEN 'gba zona sur'
            ELSE zona
        END as region,

        tipo_de_operacion, precio, moneda, expensas, ambientes,
        metros_cuadrados_totales, metros_cuadrados_cubiertos,
        antiguedad, cochera, orientacion, estado, url, fecha_publicacion,
        ROUND(precio / metros_cuadrados_totales, 2) as precio_por_m2
    FROM datos_deduplicados
    WHERE rn = 1
)

SELECT
    partido, region, tipo_de_operacion,
    precio, moneda, expensas, ambientes,
    metros_cuadrados_totales, metros_cuadrados_cubiertos,
    antiguedad, cochera, orientacion, estado, url, fecha_publicacion,
    precio_por_m2,
    'real_estate.bronze.properties_bronze' as _source_table,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM datos_finales;

##Validar datos insertados

In [0]:
%sql
select
    'real_estate.bronze.properties_bronze' as capa,
    count(*) as registros
from real_estate.bronze.properties_bronze

union all

select 
    'real_estate.silver.propiedades' as capa,
    count(*) as registros
from real_estate.silver.propiedades

##Verificar calidad de datos en Silver

In [0]:
%sql
select 
    count(*) as total_silver,
    count(case when precio <= 0 then 1 end) as precio_invalido,
    count(case when metros_cuadrados_totales <= 0 then 1 end) as metros_cuadrados_totales,
    count(case when antiguedad = 999 then 1 end) as antiguedad_placeholder_999,
    count(case when antiguedad between 0 and 200 then 1 end) as antiguedad_valida,
    count(case when partido is null or partido = '' then 1 end) as partido_vacio,
    count(case when precio_por_m2 is null then 1 end) as precio_m2_nulo,
    count(case when precio_por_m2 > 0 then 1 end) as precio_m2_valido
from real_estate.silver.propiedades

##Métricas por partido (agregaciones sobre Silver)

In [0]:
%sql
select 
    partido,
    tipo_de_operacion,
    moneda,
    count(*) as total_propiedades,
    round(avg(precio), 2) as precio_promedio,
    round(percentile(precio, 0.5), 2) as precio_mediana,
    round(min(precio), 2) as precio_minimo,
    round(max(precio), 2) as precio_maximo,
    round(avg(precio_por_m2), 2) as precio_m2_promedio,
    round(avg(metros_cuadrados_totales), 2) as m2_promedio,
    round(avg(ambientes), 1) as ambientes_promedio
from real_estate.silver.propiedades
group by partido, tipo_de_operacion, moneda
having count(*) >= 5
order by precio_promedio desc
limit 15;

##Ranking de partidos por precio

In [0]:
%sql

with precios_partido as (
    select
        partido,
        tipo_de_operacion,
        moneda,
        count(*) as total_propiedades,
        round(avg(precio), 2) as precio_promedio,
        round(avg(precio_por_m2), 2) as precio_m2_promedio
    from real_estate.silver.propiedades
    group by partido, tipo_de_operacion, moneda
    having count(*) >= 10
)
select 
    partido,
    tipo_de_operacion,
    moneda,
    total_propiedades,
    precio_promedio,
    precio_m2_promedio,
    row_number() over (partition by tipo_de_operacion, moneda order by precio_promedio desc) as ranking
from precios_partido
order by tipo_de_operacion, moneda, ranking
limit 20;

##Distribución por ambientes

In [0]:
%sql
select
    ambientes,
    tipo_de_operacion,
    moneda,
    count(*) as total_propiedades,
    round(count(*) * 100.0 / sum(count(*)) over (partition by tipo_de_operacion, moneda), 2) as pct_del_ranking,
    round(avg(precio), 2) as precio_promedio,
    round(avg(metros_cuadrados_totales), 2) as m2_promedio
from real_estate.silver.propiedades
where ambientes is not null and ambientes > 0
group by ambientes, tipo_de_operacion, moneda
order by tipo_de_operacion, moneda, ambientes

##Resumen de objetos por capa


In [0]:
%sql
select 'BRONZE' as capa, table_name, table_type
from real_estate.information_schema.tables
where table_schema = 'bronze'

union all 

select 'SILVER' as capa, table_name, table_type
from real_estate.information_schema.tables
where table_schema = 'silver'

union all

select 'GOLD' as capa, table_name, table_type
from real_estate.information_schema.tables
where table_schema = 'gold'

order by capa, table_name;

##Resumen de registros por capa

In [0]:
%sql
select 'real_estate.bronze.properties_bronze' as tabla, count(*) as registros
from real_estate.bronze.properties_bronze

union all
select 'real_estate.silver.propiedades' as capa, count(*) as registros
from real_estate.silver.propiedades;

##Encontrar registros filtrados por el pipeline

In [0]:
%sql
select 
    'precio <= 0' as motivo_filtro,
    count(*) as registros_excluidaos
from real_estate.bronze.properties_bronze
where cast(precio as double) <= 0 or precio is null

union all

select 
    'm2 <= 0' as motivo_filtro,
    count(*) as registros_excluidos
from real_estate.bronze.properties_bronze
where cast(metros_cuadrados_totales as double) <= 0 or metros_cuadrados_totales is null

union all

select 
    'tipo_operacion NULL' as motifo_filtro,
    count(*) as registros_excluidos
from real_estate.bronze.properties_bronze
where tipo_de_operacion is null

union all

select 
    'zona no mapeada' as motivo_filtro,
    count(*) as registros_excluidos
from real_estate.bronze.properties_bronze
where zona is null or zona = ''

union all

select
    'TOTAL Bronze' as motivo_filtro,
    count(*) as registros_excluidos
from real_estate.bronze.properties_bronze